# 05. DMS-Scale Discovery
Aβ42 Deep Mutational Scanning (n≈800) + ESM-2 650M + ML Composite

**Phase 1**: Seuma 2022 DMS 데이터 로드 + 기존 flow model로 trajectory 피처 추출
**Phase 2**: ESM-2 650M 임베딩 → flow model 재학습 (더 좋은 모델)
**Phase 3**: ML composite (Lasso + RF) + cross-protein transfer test

In [ ]:
# Setup
import os, sys
os.chdir('/content')
!rm -rf brain_idp_flow
!git clone https://github.com/layeredlabs06/brain-idp-flow.git brain_idp_flow
os.chdir('brain_idp_flow')
!pip install -e . -q
!pip install fair-esm scipy scikit-learn -q
sys.path.insert(0, '/content/brain_idp_flow/src')

import torch
import numpy as np
import yaml

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

---
## Phase 1: DMS Data + Flow Model Scoring
Seuma 2022 Aβ42 nucleation scores (n≈800) + trajectory features

In [ ]:
# Load DMS data
from brain_idp_flow.data.dms_loader import load_seuma_dms

os.makedirs('data/dms', exist_ok=True)
dms_data = load_seuma_dms(cache_dir='data/dms')

print(f'\nTotal mutations: {len(dms_data)}')
print(f'Nucleation score range: [{min(d["nucleation_score"] for d in dms_data):.2f}, '
      f'{max(d["nucleation_score"] for d in dms_data):.2f}]')
print(f'fAD mutations: {sum(1 for d in dms_data if d["is_fad"])}')

# Quick distribution
import matplotlib.pyplot as plt
scores = [d['nucleation_score'] for d in dms_data]
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(scores, bins=50, edgecolor='black', alpha=0.7)
ax.axvline(0, color='red', linestyle='--', label='WT')
ax.set_xlabel('Nucleation Score')
ax.set_ylabel('Count')
ax.set_title(f'Aβ42 DMS Nucleation Score Distribution (n={len(dms_data)})')
ax.legend()
fig.tight_layout()
fig.show()

In [ ]:
# Load existing flow model (35M)
import glob
from brain_idp_flow.sample import load_model
from brain_idp_flow.features.seq_embed import ESM2Embedder

with open('configs/flow.yaml') as f:
    config = yaml.safe_load(f)

ckpt_dirs = sorted(glob.glob('runs/flow/*'))
if ckpt_dirs:
    CKPT = os.path.join(ckpt_dirs[-1], 'ckpt_best.pt')
else:
    from google.colab import drive
    drive.mount('/content/drive')
    CKPT = '/content/drive/MyDrive/brain_idp_flow_ckpts/ckpt_best.pt'

model = load_model(config, CKPT, device)
embedder = ESM2Embedder(device=device)  # 35M for now
print(f'Model loaded from {CKPT}')

In [ ]:
# Score DMS mutations with flow model + trajectory features
# This is the big run: ~800 mutations × 32 ensemble samples
from brain_idp_flow.sample import sample_ensemble_with_trajectory
from brain_idp_flow.analysis.trajectory_analysis import extract_trajectory_features
from brain_idp_flow.data.dataset import ProteinEnsembleDataset
from brain_idp_flow.geometry.metrics import radius_of_gyration

ABETA_SEQ = 'DAEFRHDSGYEVHHQKLVFFAEDVGSNKGAIIGLMVGGVVIA'

# WT reference
wt_emb = embedder.embed_single(ABETA_SEQ)
wt_result = sample_ensemble_with_trajectory(
    model, wt_emb, mut_pos=0, mut_aa=0,
    n_samples=64, n_trajectory_samples=16,
    device=device,
)
wt_rg = radius_of_gyration(torch.from_numpy(wt_result['ensemble'])).mean().item()
print(f'WT Rg: {wt_rg:.2f}')

# Score all DMS mutations (batched for speed)
scored_dms = []
n_total = len(dms_data)

for i, mut in enumerate(dms_data):
    # Build mutant sequence
    mut_seq = list(ABETA_SEQ)
    mut_seq[mut['pos'] - 1] = mut['mt']
    mut_seq = ''.join(mut_seq)
    
    # ESM-2 embedding
    mut_emb = embedder.embed_single(mut_seq)
    aa_idx = ProteinEnsembleDataset.AA_TO_IDX.get(mut['mt'], 0)
    
    # Small ensemble (32) + trajectory (4) for speed
    result = sample_ensemble_with_trajectory(
        model, mut_emb,
        mut_pos=mut['pos'], mut_aa=aa_idx,
        n_samples=32, n_trajectory_samples=4,
        n_steps=50, device=device,
    )
    
    # Delta Rg
    mut_rg = radius_of_gyration(torch.from_numpy(result['ensemble'])).mean().item()
    
    # Trajectory features
    traj_feats = extract_trajectory_features(
        result['trajectory'], mutation_pos=mut['pos'] - 1,
    )
    
    scored = {
        **mut,
        'delta_rg': mut_rg - wt_rg,
        **{k: v for k, v in traj_feats.items() if not k.startswith('_')},
    }
    scored_dms.append(scored)
    
    if (i + 1) % 50 == 0:
        print(f'  Scored {i + 1}/{n_total} mutations...')

print(f'\nDone: {len(scored_dms)} mutations scored with flow model')

In [ ]:
# ESM-2 LLR for all DMS mutations
from brain_idp_flow.analysis.esm2_llr import ESM2MutationScorer

scorer = ESM2MutationScorer(device=device)

for i, mut in enumerate(scored_dms):
    llr_result = scorer.score_mutation(
        ABETA_SEQ, mut['pos'], mut['wt'], mut['mt'], fast=True,
    )
    mut['llr_site'] = llr_result['llr_site']
    
    if (i + 1) % 100 == 0:
        print(f'  LLR scored {i + 1}/{len(scored_dms)}')

print(f'LLR scoring complete')

In [ ]:
# Correlation: trajectory features vs nucleation score (n≈800!)
from scipy.stats import spearmanr
import matplotlib.pyplot as plt

ns = np.array([d['nucleation_score'] for d in scored_dms])

features_to_test = [
    ('late_velocity_site', 'Late-Stage Velocity'),
    ('velocity_variance_late', 'Velocity Variance (late)'),
    ('switching_rate_site', 'Contact Switching Rate'),
    ('delta_rg', 'Delta Rg'),
    ('llr_site', 'ESM-2 LLR'),
    ('convergence_time_site', 'Convergence Time'),
    ('contact_formation_delay', 'Contact Formation Delay'),
]

print(f'=== DMS CORRELATION (n={len(scored_dms)}) ===')
print(f'{"Feature":<30} {"rho":>8} {"p-value":>12} {"Sig?":>6}')
print('-' * 60)

for feat_key, feat_name in features_to_test:
    vals = np.array([d.get(feat_key, 0) for d in scored_dms])
    if vals.std() < 1e-10:
        continue
    rho, pval = spearmanr(vals, ns)
    sig = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else ''
    print(f'{feat_name:<30} {rho:>8.3f} {pval:>12.2e} {sig:>6}')

In [ ]:
# Key plots: trajectory features vs nucleation score
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Color by nucleation score
colors = np.array([d['nucleation_score'] for d in scored_dms])

plot_feats = [
    ('late_velocity_site', 'Late-Stage Velocity'),
    ('velocity_variance_late', 'Velocity Variance'),
    ('switching_rate_site', 'Contact Switching'),
    ('delta_rg', 'Delta Rg'),
    ('llr_site', 'ESM-2 LLR'),
    ('convergence_time_site', 'Convergence Time'),
]

for idx, (feat_key, feat_name) in enumerate(plot_feats):
    ax = axes[idx // 3][idx % 3]
    vals = np.array([d.get(feat_key, 0) for d in scored_dms])
    
    sc = ax.scatter(vals, ns, c=ns, cmap='RdBu_r', s=15, alpha=0.6, edgecolors='none')
    
    # Mark fAD mutations
    fad_idx = [i for i, d in enumerate(scored_dms) if d.get('is_fad')]
    if fad_idx:
        ax.scatter(vals[fad_idx], ns[fad_idx], s=80, facecolors='none',
                   edgecolors='black', linewidth=2, label='fAD')
    
    rho, pval = spearmanr(vals, ns)
    ax.set_xlabel(feat_name)
    ax.set_ylabel('Nucleation Score')
    ax.set_title(f'{feat_name}\nρ={rho:.3f}, p={pval:.2e}')
    if fad_idx:
        ax.legend(fontsize=8)

fig.suptitle(f'Aβ42 DMS: Flow Model Features vs Nucleation Score (n={len(scored_dms)})', fontsize=14)
fig.tight_layout()
os.makedirs('results/dms', exist_ok=True)
fig.savefig('results/dms/dms_correlations.png', dpi=150)
fig.show()

---
## Phase 2: ESM-2 650M Retraining
더 큰 language model로 flow model 재학습

In [ ]:
# Compute ESM-2 650M embeddings
from brain_idp_flow.features.seq_embed import ESM2Embedder
from brain_idp_flow.targets import load_targets

embedder_650m = ESM2Embedder(
    model_name='esm2_t33_650M_UR50D',
    device=device,
)
print(f'ESM-2 650M embed dim: {embedder_650m.embed_dim}')

targets = load_targets('configs/targets.yaml')

seq_embeddings_650m = {}
sid = 0
for tid, target in targets.items():
    emb = embedder_650m.embed_single(target.sequence)
    seq_embeddings_650m[sid] = emb.cpu()
    print(f'{tid} WT: seq_id={sid}, emb={emb.shape}')
    sid += 1
    for mut in target.mutations:
        mut_seq = target.mutant_sequence(mut)
        emb = embedder_650m.embed_single(mut_seq)
        seq_embeddings_650m[sid] = emb.cpu()
        sid += 1

torch.save(seq_embeddings_650m, 'data/seq_embeddings_650m.pt')
print(f'\n{len(seq_embeddings_650m)} embeddings saved (650M)')

In [ ]:
# Train flow model with 650M embeddings
from brain_idp_flow.train import train

with open('configs/flow_650m.yaml') as f:
    config_650m = yaml.safe_load(f)

# Build dataset if needed
if not os.path.exists('data/train.npz'):
    !python scripts/build_dataset.py --out data/train.npz --max-len 160

seq_emb_650m = torch.load('data/seq_embeddings_650m.pt', weights_only=True)

print('Starting 650M flow model training (T4: ~5-6h for 50k steps)...')
print('You can monitor with TensorBoard in the next cell.')
best_ckpt_650m = train(config_650m, seq_emb_650m, device)
print(f'\nBest checkpoint: {best_ckpt_650m}')

In [ ]:
# TensorBoard (optional)
%load_ext tensorboard
%tensorboard --logdir runs/flow_650m

In [ ]:
# Backup 650M checkpoint to Drive
import shutil
try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    backup_dir = '/content/drive/MyDrive/brain_idp_flow_ckpts_650m'
    os.makedirs(backup_dir, exist_ok=True)
    shutil.copy2(str(best_ckpt_650m), backup_dir)
    last_ckpt = str(best_ckpt_650m).replace('ckpt_best.pt', 'ckpt_last.pt')
    if os.path.exists(last_ckpt):
        shutil.copy2(last_ckpt, backup_dir)
    print(f'Backed up to {backup_dir}')
except:
    print('Drive backup skipped')

---
## Phase 3: ML Composite + Cross-Protein Transfer

In [ ]:
# Combine DMS data (Aβ42) with disease mutation data (tau + α-syn)
from brain_idp_flow.analysis.ml_predictor import run_full_ml_pipeline

# For ML, we need agg_rate as the target
# DMS data uses nucleation_score → already converted to agg_rate via exp()
# Disease mutation data uses agg_rate_relative directly

# Aβ42 DMS: already in scored_dms
print(f'Aβ42 DMS: {len(scored_dms)} mutations')

# Run ML pipeline on DMS data alone first
os.makedirs('results/ml', exist_ok=True)
dms_ml_results = run_full_ml_pipeline(
    scored_dms,
    output_dir='results/ml',
)

In [ ]:
# Load disease mutation trajectory data from 04_trajectory_discovery
# If not available, score them here
from brain_idp_flow.targets import load_targets
from brain_idp_flow.data.dataset import ProteinEnsembleDataset

targets = load_targets('configs/targets.yaml')
disease_data = []

for tid, target in targets.items():
    if tid == 'abeta42':
        continue  # Already have DMS data
    
    print(f'\nScoring {target.name}...')
    wt_emb = embedder.embed_single(target.sequence)
    wt_res = sample_ensemble_with_trajectory(
        model, wt_emb, mut_pos=0, mut_aa=0,
        n_samples=64, n_trajectory_samples=16, device=device,
    )
    wt_rg_prot = radius_of_gyration(torch.from_numpy(wt_res['ensemble'])).mean().item()
    
    for mut in target.mutations:
        mut_seq = target.mutant_sequence(mut)
        mut_emb = embedder.embed_single(mut_seq)
        aa_idx = ProteinEnsembleDataset.AA_TO_IDX.get(mut.mt, 0)
        
        res = sample_ensemble_with_trajectory(
            model, mut_emb,
            mut_pos=mut.pos, mut_aa=aa_idx,
            n_samples=32, n_trajectory_samples=4,
            device=device,
        )
        
        mut_rg = radius_of_gyration(torch.from_numpy(res['ensemble'])).mean().item()
        traj_feats = extract_trajectory_features(res['trajectory'], mutation_pos=mut.pos - 1)
        
        llr = scorer.score_mutation(target.sequence, mut.pos, mut.wt, mut.mt, fast=True)
        
        disease_data.append({
            'target': tid,
            'mutation_id': mut.id,
            'pos': mut.pos,
            'wt': mut.wt,
            'mt': mut.mt,
            'agg_rate': mut.agg_rate_relative,
            'delta_rg': mut_rg - wt_rg_prot,
            'llr_site': llr['llr_site'],
            **{k: v for k, v in traj_feats.items() if not k.startswith('_')},
        })
        print(f'  {mut.id}: done')

print(f'\nDisease mutations scored: {len(disease_data)}')

In [ ]:
# CROSS-PROTEIN TRANSFER: Train on Aβ42 DMS → Predict tau + α-syn
from brain_idp_flow.analysis.ml_predictor import run_cross_protein_transfer

# Combine all data
all_combined = scored_dms + disease_data
print(f'Combined dataset: {len(all_combined)} mutations')
print(f'  Aβ42 (DMS): {len(scored_dms)}')
print(f'  tau + α-syn (disease): {len(disease_data)}')

transfer_results = run_cross_protein_transfer(all_combined)

In [ ]:
# Save everything + Drive backup
import json

final_results = {
    'dms_n': len(scored_dms),
    'disease_n': len(disease_data),
    'ml_results': {
        'lasso_cv_rho': dms_ml_results['lasso']['cv_mean_rho'],
        'rf_cv_rho': dms_ml_results['rf']['cv_mean_rho'],
        'lasso_coefficients': dms_ml_results['lasso']['coefficients'],
        'rf_importance': dms_ml_results['rf']['feature_importance'],
    },
    'transfer_results': transfer_results,
}

with open('results/dms/dms_discovery_results.json', 'w') as f:
    json.dump(final_results, f, indent=2, default=float)

try:
    backup = '/content/drive/MyDrive/brain_idp_flow_dms_discovery'
    import shutil
    shutil.copytree('results', backup, dirs_exist_ok=True)
    print(f'Backed up to {backup}')
except:
    print('Drive backup skipped')

print('\n=== DMS DISCOVERY COMPLETE ===')